## Task 3, part 7 - Modelling of a 6th, additional model: Gaussian Process regression (protein log2FC -> RNA target)

The first non-linear model in this project. Uses the same input (each perturbation's own protein log2FC, test-time available) and the same PCA-reduced RNA target as models 3 and 4, but replaces Ridge (a *linear* map from features to target-PC scores) with Gaussian Process regression -- a kernel-based, non-linear, non-parametric model. Two reasons to reach for a GP specifically, rather than e.g. a random forest or a small neural net:

1. GPs are one of the few model families that behave sensibly with very little training data (we only have 40 training genes); trees and neural nets typically need far more data to fit anything meaningful.
2. A GP gives a genuine predictive *uncertainty* (a variance, not just a point estimate) for every prediction, for free -- something none of models 1-5 actually provided (we only ever reported the spread *across* genes after the fact, never a per-gene confidence from the model itself). This directly speaks to the assignment's ask to evaluate with "suitable metrics/uncertainty".

As in model 3, the RBF kernel's length-scale is optimized automatically by the GP itself (by maximizing the marginal likelihood during `.fit()` -- a built-in feature of GPs, no manual search needed for that part). We still choose one hyperparameter manually via the same leave-one-out cross-validation used throughout: `alpha`, the assumed observation noise added to the kernel -- conceptually the same role Ridge's alpha played, controlling how much the model trusts individual training points versus smooths over them.

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.preprocessing import StandardScaler

In [2]:
# load back in the RNA fingerprints and split prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()

selected_50 = train_40 + test_10
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

pert_FC_selected.shape

(150, 2042)

## Compute the protein log2FC "fingerprint"

Mirrors exactly what Task3_01 did for RNA and what Task3_04/Task3_05 did for the other protein-based models: for each condition, compare the mean protein expression of each perturbation's cells to that condition's control cells, on a log2 scale with a pseudocount. 4 of the 24 measured "proteins" are isotype controls and are dropped, leaving 20 real surface markers.

In [3]:
protein = sc.read_h5ad(f"{DATA_DIR}/protein.h5ad")

# drop isotype controls (antibody background-binding controls, not real markers)
isotype_controls = ["Rat_IgG2a", "Mouse_IgG1", "Mouse_IgG2a", "Mouse_IgG2b"]
protein = protein[:, ~protein.var_names.isin(isotype_controls)].copy()

# keep only control cells and cells belonging to one of the 50 selected perturbations
relevant_mask = protein.obs["perturbation"].isin(selected_50) | (protein.obs["perturbation"] == "control")
protein = protein[relevant_mask.values].copy()

# normalize (no log-transform needed here, log2FC is computed on the normalized scale directly, as in Task3_01)
sc.pp.normalize_total(protein, target_sum=1e4)
protein_norm = np.asarray(protein.X.todense())

protein.shape

/tmp/ipykernel_19534/1216798519.py:12: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(protein, target_sum=1e4)


(94502, 20)

In [4]:
PSEUDOCOUNT = 1.0

# mean normalized protein expression of control cells, per condition
control_means_protein = {}
for cond in conditions:
    mask = (protein.obs["perturbation_2"] == cond) & (protein.obs["perturbation"] == "control")
    control_means_protein[cond] = protein_norm[mask.values].mean(axis=0)

# log2FC protein fingerprint per (perturbation, condition), relative to control cells of that condition
protein_FC = {}
for pert in selected_50:
    for cond in conditions:
        mask = (protein.obs["perturbation_2"] == cond) & (protein.obs["perturbation"] == pert)
        pert_mean = protein_norm[mask.values].mean(axis=0)
        protein_FC[(pert, cond)] = np.log2((pert_mean + PSEUDOCOUNT) / (control_means_protein[cond] + PSEUDOCOUNT))

protein_FC_index = pd.MultiIndex.from_tuples(protein_FC.keys(), names=["perturbation", "condition"])
protein_FC_df = pd.DataFrame(np.vstack(list(protein_FC.values())), index=protein_FC_index, columns=protein.var_names)

protein_FC_df.shape

(150, 20)

## Reduce the target: PCA on the training RNA fingerprints

Same as models 2 and 3: fit PCA on the 40 training genes' RNA fingerprints (per condition) and predict a gene's position along those axes instead of the full ~2042-dim vector. Only ever uses training labels -- the 10 held-out genes are never involved in defining this space.

In [5]:
N_TARGET_PCS = 10

# per condition: fit PCA on the 40 training genes' RNA fingerprints, and store each training gene's score
target_pca_by_condition = {}
target_scores = {}  # (gene, condition) -> score along the target PCs, training genes only
for cond in conditions:
    train_fingerprints = np.vstack([pert_FC_selected.loc[(g, cond)].values for g in train_40])
    pca = PCA(n_components=N_TARGET_PCS, random_state=42)
    scores = pca.fit_transform(train_fingerprints)
    target_pca_by_condition[cond] = pca
    for gene, score in zip(train_40, scores):
        target_scores[(gene, cond)] = score

# how much of the training fingerprints' variance these 10 PCs capture, per condition
{cond: target_pca_by_condition[cond].explained_variance_ratio_.sum() for cond in conditions}

{'Control': np.float32(0.61914194),
 'IFNγ': np.float32(0.6695988),
 'Co-culture': np.float32(0.76921785)}

## Gaussian Process regression prediction

For a query gene in a given condition: standardize its 20-dim protein log2FC vector, fit a GP mapping protein log2FC -> the 10 RNA target-PC scores using the pool genes, predict the query's scores *and their predictive standard deviation*, then reconstruct the full RNA fingerprint (and its per-gene uncertainty) with that condition's target PCA. The pool never includes the query gene itself, so this works for both leave-one-out cross-validation and predicting the actual held-out genes.

Propagating the uncertainty from PC-space into full gene-fingerprint space uses the fact that PCA's inverse transform is linear (`fingerprint = scores @ components_ + mean_`): if the 10 PC scores have independent predictive variances, the variance of each reconstructed gene is just the sum of those variances weighted by the squared PCA loadings for that gene -- no extra model needed, just linear algebra.

In [6]:
def gp_predict(query_gene, condition, alpha, pool_genes):
    """Predict an RNA fingerprint (and its per-gene predictive std) via GP regression from protein log2FC."""
    # exclude the query gene itself from the pool used to fit the model
    fit_genes = [g for g in pool_genes if g != query_gene]

    X = np.vstack([protein_FC_df.loc[(g, condition)].values for g in fit_genes])
    y = np.vstack([target_scores[(g, condition)] for g in fit_genes])

    # standardize the protein features before the kernel sees them
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # RBF kernel: its length-scale and signal variance are optimized automatically inside .fit(),
    # by maximizing the marginal likelihood -- only alpha (assumed noise) is chosen by us, via LOOCV
    kernel = ConstantKernel(1.0, (1e-2, 1e2)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))
    gp = GaussianProcessRegressor(kernel=kernel, alpha=alpha, normalize_y=True, n_restarts_optimizer=3, random_state=42)
    gp.fit(X_scaled, y)

    query_X = scaler.transform(protein_FC_df.loc[(query_gene, condition)].values.reshape(1, -1))
    pred_scores, pred_std = gp.predict(query_X, return_std=True)

    # reconstruct the full RNA fingerprint from the predicted target-PC scores
    pca = target_pca_by_condition[condition]
    pred_fingerprint = pca.inverse_transform(pred_scores)[0]

    # propagate per-PC predictive variance into per-gene predictive variance (see markdown above),
    # then take the square root to get a per-gene predictive standard deviation
    pred_fingerprint_std = np.sqrt((pred_std[0] ** 2) @ (pca.components_ ** 2))

    return pred_fingerprint, pred_fingerprint_std

## Choosing the noise level (alpha) via leave-one-out cross-validation

Hold out one training gene at a time, predict it from the other 39 (per condition), and compare a few candidate alpha values using prediction accuracy (MSE) -- the same selection criterion used for every other model, so results stay comparable. This never touches the 10 held-out test genes.

In [7]:
candidate_alphas = [1e-6, 1e-4, 1e-2, 0.1, 1, 10]

cv_mse_by_alpha = {}
for alpha in candidate_alphas:
    squared_errors = []
    for cond in conditions:
        for gene in train_40:
            # gp_predict excludes the query gene itself from the pool, so this is a genuine leave-one-out prediction
            pred, _ = gp_predict(gene, cond, alpha, train_40)
            true = pert_FC_selected.loc[(gene, cond)].values
            squared_errors.append(np.mean((true - pred) ** 2))
    cv_mse_by_alpha[alpha] = np.mean(squared_errors)

best_alpha = min(cv_mse_by_alpha, key=cv_mse_by_alpha.get)
cv_mse_by_alpha, best_alpha

/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


...

/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda

/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda

/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda

/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda

({1e-06: np.float64(0.0021944887511412344),
  0.0001: np.float64(0.0021944884534851033),
  0.01: np.float64(0.002194428333657904),
  0.1: np.float64(0.002193832676505175),
  1: np.float64(0.002188156501811771),
  10: np.float64(0.0021882838626903667)},
 1)

## Predict the held-out test genes and evaluate

Use the chosen alpha to predict each of the 10 held-out genes' RNA fingerprint (and its predictive uncertainty) from their own protein log2FC, then evaluate with the same metrics used for the other models so results are directly comparable. Also record each gene's average predictive std, to check afterwards whether the model's own self-reported confidence actually tracks its real accuracy.

In [8]:
# predict each held-out (gene, condition) pair from the 40 training genes, using the CV-chosen alpha
gp_predictions = {}
gp_pred_std = {}
for cond in conditions:
    for gene in test_10:
        pred, pred_std = gp_predict(gene, cond, best_alpha, train_40)
        gp_predictions[(gene, cond)] = pred
        gp_pred_std[(gene, cond)] = pred_std


def evaluate_predictions(true_df, predictions_by_row):
    """Compare each true fingerprint against its predicted fingerprint (looked up per row)."""
    records = []
    for (pert, cond), true_fc in true_df.iterrows():
        pred_fc = predictions_by_row[(pert, cond)]
        pearson_r, _ = pearsonr(true_fc, pred_fc)
        spearman_r, _ = spearmanr(true_fc, pred_fc)
        mse = np.mean((true_fc - pred_fc) ** 2)
        records.append({
            "perturbation": pert,
            "condition": cond,
            "pearson_r": pearson_r,
            "spearman_r": spearman_r,
            "mse": mse,
            # average predictive std across all ~2042 genes, as a single "how confident was the model" number
            "mean_pred_std": gp_pred_std[(pert, cond)].mean(),
        })
    return pd.DataFrame(records)


gp_eval = evaluate_predictions(pert_FC_selected.loc[test_10, :], gp_predictions)
gp_eval

/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda

/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda

/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda

/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda3/envs/denbi/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/ubuntu/miniconda

,perturbation,condition,pearson_r,spearman_r,mse,mean_pred_std
0,KCNN4,Control,0.828347,0.457209,0.001508,0.001350
1,KCNN4,IFNγ,0.845401,0.398457,0.001093,0.001890
2,KCNN4,Co-culture,0.862499,0.339076,0.001128,0.001373
3,TIMM50,Control,0.741629,0.392909,0.002767,0.001350
4,TIMM50,IFNγ,0.625137,0.294633,0.003312,0.001901
5,TIMM50,Co-culture,0.698907,0.194892,0.003830,0.001373
6,TXNDC17,Control,0.805104,0.492349,0.004353,0.001350
7,TXNDC17,IFNγ,0.620815,0.445548,0.004326,0.001901
8,TXNDC17,Co-culture,0.762693,0.396490,0.004462,0.001373
9,CORO1A,Control,0.806770,0.375206,0.001195,0.001349


## Is the model's uncertainty actually informative?

A well-calibrated uncertainty estimate should be *higher* for genes the model gets more wrong. Check this directly: correlate each test gene's average predictive std against its actual error (using `1 - pearson_r`, so higher = worse, in the same direction as std).

In [9]:
uncertainty_calibration_r, uncertainty_calibration_p = pearsonr(gp_eval["mean_pred_std"], 1 - gp_eval["pearson_r"])
uncertainty_calibration_r, uncertainty_calibration_p

(np.float64(-0.053699161699409935), np.float64(0.7780754069755438))

In [10]:
metrics = ["pearson_r", "spearman_r", "mse"]

# per-condition breakdown (n=10 genes each) -- for biological interpretation
per_condition = gp_eval.groupby("condition")[metrics].agg(["mean", "std"])

# pooled across all held-out (gene, condition) pairs (n=30) -- single headline number, comparable to the other models
overall = gp_eval[metrics].agg(["mean", "std"])

per_condition

pearson_r           spearman_r                 mse          
                mean       std       mean       std      mean       std
condition                                                              
Co-culture  0.592829  0.498193   0.267086  0.107349  0.003967  0.005891
Control     0.754667  0.137725   0.398938  0.117841  0.002500  0.001594
IFNγ        0.717778  0.254574   0.379546  0.096302  0.002429  0.001927

In [11]:
overall

,pearson_r,spearman_r,mse
mean,0.688425,0.348523,0.002966
std,0.328614,0.119412,0.003637


## Discussion (initial draft -- please rewrite)

**What this notebook does:** The first non-linear model in the project: Gaussian Process regression mapping each perturbation's protein log2FC to the 10 RNA target-PC scores (same target reduction as models 2 and 3). The RBF kernel's length-scale and signal variance are optimized automatically inside `.fit()` by maximizing the marginal likelihood; only the noise level (`alpha`) is chosen manually, via the same leave-one-gene-out CV used throughout. Unlike every previous model, this one also returns a genuine per-gene predictive standard deviation, not just a point estimate.

**Results:** the LOOCV curve is nearly flat -- MSE barely moves from 0.0021945 (alpha=1e-6) down to 0.0021882 (alpha=1, the chosen value) and back up to 0.0021883 (alpha=10), a difference of well under 1%. Final test performance: Pearson r = 0.688, Spearman r = 0.349, MSE = 0.00297 -- again statistically indistinguishable from the baseline and every other protein-based model. The kernel optimizer also repeatedly hit its length-scale/constant-value bounds during fitting (the `ConvergenceWarning`s above), which is itself informative: it means the optimizer kept wanting to push toward "treat every training gene as roughly equally relevant", i.e. toward the same near-uniform-average behavior Ridge converged to in models 2/3 at very high alpha. The non-linear kernel had the flexibility to find something more structured than a straight line, and didn't -- reasonably strong evidence that the ceiling here isn't model flexibility, it's the amount of information actually present in protein log2FC + 40 genes.

**Uncertainty check:** we tested whether the GP's own predictive std is actually informative, by correlating each test gene's average predictive std against its real error (1 - Pearson r). Result: r = -0.054, p = 0.78 -- no meaningful relationship, and if anything the wrong sign. In other words, the model's confidence estimates are not well-calibrated: it isn't more "unsure" about the genes it actually gets wrong. This is a legitimate and worth-reporting limitation rather than a reason to distrust the whole exercise -- with only 39 training points per fit and a single shared kernel across all 10 target dimensions, the GP simply doesn't have enough data to learn a query-dependent notion of its own uncertainty here.

**Overall:** across 6 models now, the two that use this experiment's own protein readout with a flexible learner (Ridge in model 3, GP here) both find only a sliver of real signal (a shallow but genuine interior CV optimum), and neither translates into a test-set improvement over the plain baseline. That's a consistent, honest conclusion: the bottleneck is signal-to-noise and sample size (40 genes), not the choice of algorithm.